In [ ]:
# ============================================================
# TIC 138829403 — PROPIEDADES DOMINANTES DESDE MASSAGE
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

T_SUN = 5772.0

tic_id = "TIC 138829403"

# ------------------------------------------------------------
# Localizar la estrella en el catálogo
# ------------------------------------------------------------

catalogo = pd.read_csv(
    "/home/luciam/Documents/starry/"
    "Resultados_Orion/catalogo_completo_mist_sommers.csv"
)

tic_num = int(tic_id.replace("TIC ", ""))

star = catalogo.loc[
    catalogo["TIC"] == tic_num
]

if len(star) == 0:
    raise ValueError(f"{tic_id} no encontrado en el catálogo.")

star = star.iloc[0]

# ------------------------------------------------------------
# MASSAGE MIST
# ------------------------------------------------------------

T_dom = float(star["T_median_1"])
sigma_T_dom = float(star["T_std_1"])

M_dom = float(star["mass_median_1"])
sigma_M_dom = float(star["mass_std_1"])

logL_dom = float(star["lgL_median_1"])
sigma_logL_dom = float(star["lgL_std_1"])

age_dom = float(star["age_median_1"])
sigma_age_dom = float(star["age_std_1"])

# ------------------------------------------------------------
# Radio derivado
# ------------------------------------------------------------

L_dom = 10**logL_dom

R_dom = (
    np.sqrt(L_dom)
    * (T_dom / T_SUN)**(-2)
)

sigma_R_dom = R_dom * np.sqrt(
    (0.5 * np.log(10) * sigma_logL_dom)**2
    +
    (2 * sigma_T_dom / T_dom)**2
)

print("=" * 70)
print(tic_id)
print("=" * 70)

print(f"T_dom       = {T_dom:.2f} ± {sigma_T_dom:.2f} K")
print(f"M_dom       = {M_dom:.4f} ± {sigma_M_dom:.4f} Msun")
print(f"log L_dom   = {logL_dom:.4f} ± {sigma_logL_dom:.4f}")
print(f"L_dom       = {L_dom:.4f} Lsun")
print(f"R_dom       = {R_dom:.4f} ± {sigma_R_dom:.4f} Rsun")
print(f"Age_dom     = {age_dom:.3f} ± {sigma_age_dom:.3f} Myr")

In [ ]:
# ============================================================
# CARGAR MIST Y SELECCIONAR PRE-MS
# ============================================================

path_modelos = Path(
    "/home/luciam/Documents/starry/"
    "Resultados_Orion/datos_estrellas_massage_orion/modelos"
)

mist = pd.read_csv(path_modelos / "MIST_V2.csv")

# Solamente fase joven / pre-MS
mist_pms = mist.loc[
    mist["phase"] == -1
].copy()

# ------------------------------------------------------------
# Convertir cantidades a unidades físicas
# ------------------------------------------------------------

mist_pms["Age_Myr"] = mist_pms["star_age"] / 1e6
mist_pms["Teff_K"] = 10**mist_pms["log_Teff"]
mist_pms["L_Lsun"] = 10**mist_pms["log_L"]
mist_pms["R_Rsun"] = 10**mist_pms["log_R"]

print("=" * 70)
print("MIST PRE-MS")
print("=" * 70)

print(f"Puntos originales : {len(mist):,}")
print(f"Puntos pre-MS     : {len(mist_pms):,}")

print(
    f"Edad: "
    f"{mist_pms['Age_Myr'].min():.5f} – "
    f"{mist_pms['Age_Myr'].max():.2f} Myr"
)

print(
    f"Masa: "
    f"{mist_pms['star_mass'].min():.3f} – "
    f"{mist_pms['star_mass'].max():.3f} Msun"
)

In [ ]:
# ============================================================
# AJUSTE MIST — COMPONENTE DOMINANTE
# ============================================================

fit = mist_pms.copy()

# ------------------------------------------------------------
# Chi-square
# ------------------------------------------------------------

fit["chi2_T"] = (
    (fit["Teff_K"] - T_dom) / sigma_T_dom
)**2

fit["chi2_L"] = (
    (fit["log_L"] - logL_dom) / sigma_logL_dom
)**2

fit["chi2_M"] = (
    (fit["star_mass"] - M_dom) / sigma_M_dom
)**2

fit["chi2"] = (
    fit["chi2_T"]
    + fit["chi2_L"]
    + fit["chi2_M"]
)

# ------------------------------------------------------------
# Mejor modelo
# ------------------------------------------------------------

best_idx = fit["chi2"].idxmin()
best = fit.loc[best_idx]

print("=" * 70)
print("MEJOR MODELO MIST — COMPONENTE DOMINANTE")
print("=" * 70)

print(f"Edad       = {best['Age_Myr']:.5f} Myr")
print(f"Masa       = {best['star_mass']:.6f} Msun")
print(f"Teff       = {best['Teff_K']:.2f} K")
print(f"L          = {best['L_Lsun']:.6f} Lsun")
print(f"R          = {best['R_Rsun']:.6f} Rsun")
print(f"log g      = {best['log_g']:.4f}")
print(f"log L      = {best['log_L']:.4f}")
print(f"chi2       = {best['chi2']:.4f}")

print()
print("Componentes de chi-square:")
print(f"  chi2_T = {best['chi2_T']:.4f}")
print(f"  chi2_L = {best['chi2_L']:.4f}")
print(f"  chi2_M = {best['chi2_M']:.4f}")

In [ ]:
# ============================================================
# REGIÓN CERCANA AL MEJOR AJUSTE
# ============================================================

delta_chi2_max = 20

region = fit.loc[
    fit["chi2"] <= best["chi2"] + delta_chi2_max
].copy()

print(f"Modelos dentro de Δχ² ≤ {delta_chi2_max}: {len(region):,}")

print("\nRango de masas:")
print(
    region["star_mass"].min(),
    "→",
    region["star_mass"].max()
)

print("\nRango de edades:")
print(
    region["Age_Myr"].min(),
    "→",
    region["Age_Myr"].max()
)

In [ ]:
# ============================================================
# EDAD ADOPTADA DEL SISTEMA — MASSAGE
# ============================================================

age_sys_Myr = float(star["age_median_1"])
sigma_age_sys_Myr = float(star["age_std_1"])

# MASSAGE: edad en Myr
# Isócronas: log10(edad / años)

logAge_sys = np.log10(age_sys_Myr * 1e6)

print("=" * 65)
print("EDAD ADOPTADA — MASSAGE")
print("=" * 65)
print(f"Edad MASSAGE = {age_sys_Myr:.4f} ± {sigma_age_sys_Myr:.4f} Myr")
print(f"logAge        = {logAge_sys:.5f}")
print("=" * 65)

In [ ]:
print("=" * 65)
print("COMPROBACIÓN DE LA EDAD")
print("=" * 65)

print(f"Edad adoptada      = {age_sys_Myr:.3f} Myr")
print(f"logAge adoptado    = {logAge_sys:.5f}")
print(
    f"Rango isócronas    = "
    f"{iso_all['logAge'].min():.5f} – "
    f"{iso_all['logAge'].max():.5f}"
)

if iso_all["logAge"].min() <= logAge_sys <= iso_all["logAge"].max():
    print("✓ La edad está dentro del rango de las isócronas.")
else:
    print("⚠ La edad está fuera del rango de las isócronas.")

In [ ]:
# ============================================================
# MASA DE LA SECUNDARIA A LA EDAD ADOPTADA
# ============================================================
m_min = iso_all["Mass"].min()
m_max = iso_all["Mass"].max()

m_grid = np.linspace(m_min, m_max, 2000)

r_model = interp_R(logAge_sys, m_grid)

valid = np.isfinite(r_model)

m_valid = m_grid[valid]
r_valid = r_model[valid]

if len(m_valid) == 0:
    raise ValueError(
        "No existen modelos válidos a la edad adoptada."
    )

# Masa cuyo radio reproduce mejor el radio observado
idx = np.argmin(np.abs(r_valid - R2))

M2_est = m_valid[idx]
R2_model = r_valid[idx]

# Propiedades de la secundaria
Te2 = float(interp_Teff(logAge_sys, M2_est))
logL2 = float(interp_logL(logAge_sys, M2_est))
L2 = 10**logL2

print("=" * 65)
print("SECUNDARIA — BHAC15")
print("=" * 65)
print(f"Edad adoptada = {age_sys_Myr:.3f} Myr")
print(f"M2            = {M2_est:.5f} Msun")
print(f"R2 modelo     = {R2_model:.5f} Rsun")
print(f"R2 observado  = {R2:.5f} Rsun")
print(f"Teff2         = {Te2:.2f} K")
print(f"logL2         = {logL2:.5f}")
print(f"L2            = {L2:.5f} Lsun")
print("=" * 65)